##### Imports

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [2]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = Ollama(model="gemma3:12b", request_timeout=360.0)

/opt/miniconda3/envs/cpanagiotop/lib/python3.11/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


##### Load Data

In [18]:
documents = SimpleDirectoryReader("cases_eval/case_002").load_data()

In [11]:
print(f"Loaded {len(documents)} document(s)")

Loaded 4 document(s)


In [19]:
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

In [6]:
llm = Settings.llm

##### Test Question(s)

In [7]:
response = llm.complete("Which country is referenced in the SAR report?")
print(response)

The SAR (Suspicious Activity Report) frequently references **China**.

Here's why:

*   **Money Laundering Concerns:** SARs are often filed due to concerns about money laundering, and China is a significant area of focus for anti-money laundering efforts globally due to its large economy, complex financial system, and potential for illicit financial flows.
*   **Trade-Based Money Laundering:** SARs often involve trade-based money laundering schemes, which frequently involve transactions with or through Chinese entities.
*   **Real Estate:** Purchases of real estate in various countries, sometimes involving shell companies linked to China, are common triggers for SAR filings.
*   **Cryptocurrency:** Transactions involving cryptocurrency, which often originate or pass through China, are also frequently flagged in SARs.



**Please note:** SARs are confidential, so the specific details of which transactions are reported are not publicly available. However, based on publicly available info

In [20]:
response = query_engine.query("Which country is referenced in the email?")
print(response)

I am sorry, I cannot answer this question. The provided information does not contain any references to countries.


In [21]:
response = llm.complete("Which companies are mentioned both in narrative files (emails or promotional text) and in the bank statements?")
print(response)

Okay, I understand the task. To figure out which companies are mentioned in both the narrative files (emails/promotional text) and the bank statements, I need the content of both.

**Here's the process I'll follow and what I need from you:**

1. **You Provide the Data:**  You need to give me the content of:
   *   **Narrative Files:**  This includes the text from emails, promotional materials, or any other narrative documents.  Please provide this as text (copy and paste is fine, or upload a file if it's a large amount).
   *   **Bank Statements:** This includes the descriptions of transactions from the bank statements. Please provide this as text as well.  I need the transaction descriptions (e.g., "Amazon.com", "Target", "Payment to Acme Corp").

2. **Data Cleaning and Preparation (I'll do this):**
   *   **Lowercase Conversion:** I'll convert all text to lowercase to ensure consistent matching (e.g., "Amazon" will match "amazon").
   *   **Punctuation Removal:** I'll remove punctuat

In [22]:
response = query_engine.query("Which companies are mentioned both in narrative files (emails or promotional text) and in the bank statements?")
print(response)

Horizon Global Ltd is mentioned in both the bank statements.


#### Evaluation

Q1: Which companies are mentioned both in narrative files (emails or promotional text) and in the bank statements?

_Reformed to:_ Is there at least one company appearing both in narrative files and in bank statements?

Ground truth: **No**

In [27]:
QUESTION_Q1 = """
Is there at least one company that appears both in narrative files
(emails, promo texts, image descriptions) and in the bank statements?

Answer only with "Entity1, Entity2".
"""

In [28]:
response = query_engine.query(QUESTION_Q1)
model_answer = str(response).strip()
print(model_answer)

Horizon Global Ltd


In [25]:
QUESTION_Q2 = """
Is there at least one company that appears both in narrative files
(emails, promo texts, image descriptions) and in the bank statements?

If Yes, answer only naming the company.
"""

In [26]:
response = query_engine.query(QUESTION_Q2)
model_answer = str(response).strip()
print(model_answer)

Horizon Global Ltd
